In [2]:
!git clone https://github.com/sscardapane/broken-repo.git

Cloning into 'broken-repo'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 12 (delta 0), reused 12 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), done.


In [3]:
%cd broken-repo

/content/broken-repo


In [4]:
!python3 -m unittest discover -s tests

F..F.
FAIL: test_completed_task_disappears_from_default_list (test_cli.CliTest.test_completed_task_disappears_from_default_list)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/content/broken-repo/tests/test_cli.py", line 35, in test_completed_task_disappears_from_default_list
    self.assertIn("No tasks.", list_result.stdout)
AssertionError: 'No tasks.' not found in '1. [ ] Fix failing tests\n'

FAIL: test_complete_persists_done_status (test_store.TaskStoreTest.test_complete_persists_done_status)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/content/broken-repo/tests/test_store.py", line 32, in test_complete_persists_done_status
    self.assertTrue(reloaded.get(1).done)
AssertionError: False is not true

----------------------------------------------------------------------
Ran 5 tests in 0.390s

FAILED (failures=2)


## Minimal local LLM client

For Colab with a T4 GPU, run the next two cells to serve `Qwen/Qwen3.5-4B` inside the Colab VM. The OpenAI client below talks to `127.0.0.1`, so inference stays on the remote VM and does not use a hosted API.

For macOS, the simplest path is Ollama with a smaller coding model. In a terminal, run:

```bash
ollama pull qwen2.5-coder:1.5b
```

Then skip the Colab serving cell and use the Ollama `BASE_URL`/`MODEL` lines below.

In [ ]:
# Colab/T4 only: install the client and the lightweight Transformers server.
# This can take a few minutes because Qwen3.5 currently needs recent Transformers support.
!pip -q install -U openai pillow torchvision
!pip -q install -U "transformers[serving] @ git+https://github.com/huggingface/transformers.git@main"

In [ ]:
# Colab/T4 only: start an OpenAI-compatible server inside this VM.
# The client below calls 127.0.0.1, so generation happens on the Colab runtime.
import subprocess
import time
import urllib.request

BASE_URL = "http://127.0.0.1:8000/v1"
MODEL = "Qwen/Qwen3.5-4B"

server = subprocess.Popen(
    [
        "transformers",
        "serve",
        "--force-model",
        MODEL,
        "--port",
        "8000",
        "--continuous-batching",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

for _ in range(120):
    try:
        urllib.request.urlopen(f"{BASE_URL}/models", timeout=2)
        break
    except Exception:
        time.sleep(5)
else:
    raise RuntimeError("The model server did not become ready in time.")

print("Model server is ready.")

In [ ]:
from openai import OpenAI

# macOS/Ollama alternative:
# BASE_URL = "http://127.0.0.1:11434/v1"
# MODEL = "qwen2.5-coder:1.5b"

client = OpenAI(base_url=BASE_URL, api_key="EMPTY")


def llm(prompt, system="You are a concise coding assistant.", max_tokens=512):
    kwargs = {}
    if "Qwen3.5" in MODEL:
        kwargs["extra_body"] = {"chat_template_kwargs": {"enable_thinking": False}}

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
        max_tokens=max_tokens,
        **kwargs,
    )
    return response.choices[0].message.content


print(llm("In one sentence, say what a failing unit test is."))